In [1]:
import pandas as pd
from pathlib import Path

ESM_CSV = Path("./carbon_density_AR_ESM_global_paper.csv")
IAM_CSV = Path("./carbon_density_AR_IAM_global_paper.csv")

esm_df = pd.read_csv(ESM_CSV)
iam_df = pd.read_csv(IAM_CSV)

# Print columns
print("ESM columns:", esm_df.columns.tolist())
print("IAM columns:", iam_df.columns.tolist())

# Map ssp126 -> SSP1-26, ssp245 -> SSP2-45, etc.
ssp_map = {
    'ssp126': 'SSP1-26',
    'ssp245': 'SSP2-45',
    'ssp370': 'SSP3-70',
    'ssp585': 'SSP5-85'
}
esm_df['SSP_std'] = esm_df['SSP'].map(ssp_map)
iam_df['SSP_std'] = iam_df['Scenario'].str.strip()

# Rename columns for merging
iam_df = iam_df.rename(columns={
    "Carbon_Density_IAM_TCO2_ha": "IAM_CarbonDensity",
    "Model": "IAM_model"
})

# Merge on standardized SSP
merged = esm_df.merge(
    iam_df[["IAM_model", "SSP_std", "IAM_CarbonDensity"]],
    left_on="SSP_std",
    right_on="SSP_std",
    how="inner"
)

merged["scaling_factor"] = merged["carbon_density_ESM"] / merged["IAM_CarbonDensity"]

# Save to Excel
output_path = "scaling_factors_AR_global_paper.xlsx"
merged.to_excel(output_path, index=False)

print(f"Saved scaling factors to {output_path}")

ESM columns: ['Model', 'ESM', 'SSP', 'Transition', 'carbon_density_ESM', 'LastYear']
IAM columns: ['Scenario', 'Model', 'Carbon_Density_IAM_TCO2_ha']
Saved scaling factors to scaling_factors_AR_global_paper.xlsx
